In [1]:
import pandas as pd
import os
import pytz
import statistics
from datetime import datetime
from datetime import timedelta
import numpy as np
import pthelperfunctions as helper

pd.set_option('display.max_rows', None)

#path_mob = "/gpfs/gibbs/project/jain_snigdha/shared/EM_PTOT_project/mobilization_output"
#path_ptot = "/gpfs/gibbs/project/jain_snigdha/shared/EM_PTOT_project/ot_pt_output"

In [2]:
#Load block data from CLIF-Mobility output.
path = os.path.join(helper.path_out, "block_compiled_3.parquet")
block_df = pd.read_parquet(path)

#Load the CSV with the columns we want
path = os.path.join(helper.path_out, "items_to_look.csv")
items_df = pd.read_csv(path)
items_chart_df = items_df[items_df['linksto']=='chartevents']
items_proc_df = items_df[items_df['linksto']=='procedureevents']

In [3]:
#Load mimic chart events
chart_df = helper.load_data("mimic","chartevents", folder='icu')
chart_df.rename(columns={'subject_id': 'patient_id','hadm_id':'hospitalization_id'}, inplace=True)

#Filter for our cohort and needs
chart_mask = \
        chart_df["patient_id"].isin(block_df['patient_id'].astype(int)) &\
        chart_df["itemid"].isin(items_chart_df['itemid']) &\
        chart_df["value"].notna()

filtered_chart_df = chart_df[chart_mask]

path = os.path.join(helper.path_out, "chartevents_filtered.parquet")
filtered_chart_df.to_parquet(path)

#Print DF
print(f"Length: {len(filtered_chart_df)}")
print("Columns")
filtered_chart_df.dtypes
filtered_chart_df.head(30)

Length: 6708871
Columns


,patient_id,hospitalization_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
9259,10001884,26184834,37510196,2091.0,2131-01-18 05:18:00,2131-01-18 05:19:00,224086,Good,NaN,None,0.0
9262,10001884,26184834,37510196,2091.0,2131-01-18 05:18:00,2131-01-18 05:19:00,229321,2a - Passive or Active ROM,2.0,None,0.0
9329,10001884,26184834,37510196,3178.0,2131-01-11 07:00:47,None,220001,.Care Plan - Impaired Tissue Perfusion,NaN,None,NaN
9602,10001884,26184834,37510196,6823.0,2131-01-15 21:00:00,2131-01-15 22:07:00,224086,Tolerated Well,NaN,None,0.0
9610,10001884,26184834,37510196,6823.0,2131-01-15 21:00:00,2131-01-15 22:07:00,229321,2b - Turning in bed,2.0,None,0.0
9781,10001884,26184834,37510196,6823.0,2131-01-16 00:00:00,2131-01-16 01:20:00,227341,No,0.0,None,0.0
9818,10001884,26184834,37510196,6823.0,2131-01-16 00:00:00,2131-01-16 01:21:00,224086,Tolerated Well,NaN,None,0.0
9826,10001884,26184834,37510196,6823.0,2131-01-16 00:00:00,2131-01-16 01:21:00,229321,2b - Turning in bed,2.0,None,0.0
9983,10001884,26184834,37510196,6823.0,2131-01-16 03:00:00,2131-01-16 05:25:00,224086,Tolerated Well,NaN,None,0.0
9988,10001884,26184834,37510196,6823.0,2131-01-16 03:00:00,2131-01-16 05:25:00,229321,2b - Turning in bed,2.0,None,0.0


chartevents
category = Treatments
labels =
    Activity / Mobility (JH-HLM)
    Range of Motion Status
    Basic Mobility (AM-PAC)    Daily Activity (AM-PAC1        
Applied Cognitive (AM-PA     1    
Test Performed (AM-P          1
Treatments (AM-           1
Discharge Recommendations         Activity / Mobility (RN Daily Mobility Goal)

chartevents
category = Adm History/FHPA
labels =
    Past medical history
    Self ADL
    History of slips / falls
    Balance
    Judgement
    Use of assistive devices
    Pressure Ulcer Present

chartevents
category = Restraint/Support Systems
label = 
    History of falling (within 3 mnths)

chartevents
category = General
labels =
    Problem List

chartevents
category = MD Progress Note
label = 
    Mobilization Plan
    Sedation Goal
    Delirium
    Disposition

procedureevents
category = 3-Significant Events
Label =
    Fall
    Unplanned Line/Catheter Removal (Patient InitPAC)